# 04 - SQL Business Analysis

## Objective

The objective of this notebook is to validate and deepen the insights found during the Exploratory Data Analysis phase using SQL.

This phase focuses on answering business questions with structured SQL queries.

The goal is to demonstrate practical SQL skills such as:

- `JOIN`
- `GROUP BY`
- `CASE WHEN`
- Common Table Expressions (`CTE`)
- Window Functions
- Ranking
- Running totals
- Year-over-year growth
- Pareto analysis

This notebook uses the cleaned dataset generated in `02_data_cleaning.ipynb`.

## Business Problem Reminder

Nova Retail has strong sales, but profitability does not always grow at the same pace.

The main question remains:

> **How can Nova Retail improve profitability without simply increasing sales?**

## How to Read This Notebook

Each section follows the same logic:

1. **Business question**: what decision are we trying to support?
2. **SQL query**: how do we answer the question with SQL?
3. **Business interpretation**: what should we look for in the result?

This notebook is designed to be readable by both technical and non-technical audiences.

## 1. Notebook Setup

This section imports the required libraries and defines helper functions for running SQL queries.

In [6]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


def format_currency(value):
    if pd.isna(value):
        return "N/A"
    return f"${value:,.0f}"


def format_percent(value):
    if pd.isna(value):
        return "N/A"
    return f"{value:.2%}"


def run_query(query, connection, title=None):
    """Run a SQL query and display the result as a DataFrame."""
    if title:
        print(title)
    result = pd.read_sql_query(query, connection)
    display(result)
    return result

## 2. Load the Cleaned Dataset

The SQL phase should use the cleaned dataset:

```text
data/cleaned/cleaned_superstore.csv
```

If this file is not found, the notebook can fall back to the raw dataset for demonstration purposes and apply the same cleaning logic.

In [7]:
possible_cleaned_paths = [
    Path("cleaned_superstore.csv")
]

possible_raw_paths = [
    Path("superstore.csv")
]

DATA_PATH = None
DATA_SOURCE = None

for path in possible_cleaned_paths:
    if path.exists():
        DATA_PATH = path
        DATA_SOURCE = "cleaned"
        break

if DATA_PATH is None:
    for path in possible_raw_paths:
        if path.exists():
            DATA_PATH = path
            DATA_SOURCE = "raw"
            break

if DATA_PATH is None:
    raise FileNotFoundError(
        "No dataset was found. Please make sure cleaned_superstore.csv or superstore.csv is available."
    )

print(f"Dataset loaded from: {DATA_PATH}")
print(f"Data source type: {DATA_SOURCE}")

df = pd.read_csv(DATA_PATH)
display(df.head())

Dataset loaded from: cleaned_superstore.csv
Data source type: cleaned


,row_id,order_id,order_date,ship_date,order_year,order_month,order_month_name,order_quarter,order_year_month,order_week,order_priority,customer_id,customer_name,segment,product_id,product_name,category,sub_category,country,state,city,region,market,market_group,sales,quantity,discount,profit,shipping_cost,ship_mode,shipping_delay_days,profit_margin,profit_status,discount_band,shipping_cost_ratio,order_size
0,36624,CA-2011-130813,2011-01-07,2011-01-09,2011,1,January,Q1,2011-01,2,High,LS-172304,Lycoris Saunders,Consumer,OFF-PA-10002005,Xerox 225,Office Supplies,Paper,United States,California,Los Angeles,West,US,North America,19,3,0.00,9.33,4.37,Second Class,2,0.49,Profitable,No discount,0.23,Medium
1,37033,CA-2011-148614,2011-01-21,2011-01-26,2011,1,January,Q1,2011-01,4,Medium,MV-174854,Mark Van Huff,Consumer,OFF-PA-10002893,"Wirebound Service Call Books, 5 1/2"" x 4""",Office Supplies,Paper,United States,California,Los Angeles,West,US,North America,19,2,0.00,9.29,0.94,Standard Class,5,0.49,Profitable,No discount,0.05,Small
2,31468,CA-2011-118962,2011-08-05,2011-08-09,2011,8,August,Q3,2011-08,32,Medium,CS-121304,Chad Sievert,Consumer,OFF-PA-10000659,"Adams Phone Message Book, Professional, 400 Me...",Office Supplies,Paper,United States,California,Los Angeles,West,US,North America,21,3,0.00,9.84,1.81,Standard Class,4,0.47,Profitable,No discount,0.09,Medium
3,31469,CA-2011-118962,2011-08-05,2011-08-09,2011,8,August,Q3,2011-08,32,Medium,CS-121304,Chad Sievert,Consumer,OFF-PA-10001144,Xerox 1913,Office Supplies,Paper,United States,California,Los Angeles,West,US,North America,111,2,0.00,53.26,4.59,Standard Class,4,0.48,Profitable,No discount,0.04,Small
4,32440,CA-2011-146969,2011-09-29,2011-10-03,2011,9,September,Q3,2011-09,40,High,AP-109154,Arthur Prichep,Consumer,OFF-PA-10002105,Xerox 223,Office Supplies,Paper,United States,California,Los Angeles,West,US,North America,6,1,0.00,3.11,1.32,Standard Class,4,0.52,Profitable,No discount,0.22,Small


## 3. Prepare Dataset for SQL

If the cleaned dataset is already loaded, this section only checks that required columns exist.

If the raw dataset is loaded, the necessary cleaning steps are applied so that the SQL analysis can still run.

In [8]:
def prepare_superstore_data(data, source_type):
    data = data.copy()

    if source_type == "raw":
        rename_columns = {
            "Category": "category",
            "City": "city",
            "Country": "country",
            "Customer.ID": "customer_id",
            "Customer.Name": "customer_name",
            "Discount": "discount",
            "Market": "market",
            "记录数": "record_count",
            "Order.Date": "order_date",
            "Order.ID": "order_id",
            "Order.Priority": "order_priority",
            "Product.ID": "product_id",
            "Product.Name": "product_name",
            "Profit": "profit",
            "Quantity": "quantity",
            "Region": "region",
            "Row.ID": "row_id",
            "Sales": "sales",
            "Segment": "segment",
            "Ship.Date": "ship_date",
            "Ship.Mode": "ship_mode",
            "Shipping.Cost": "shipping_cost",
            "State": "state",
            "Sub.Category": "sub_category",
            "Year": "order_year",
            "Market2": "market_group",
            "weeknum": "order_week"
        }
        data = data.rename(columns=rename_columns)

        if "record_count" in data.columns:
            data = data.drop(columns=["record_count"])

    # Clean text values
    text_columns = data.select_dtypes(include="object").columns
    for col in text_columns:
        data[col] = data[col].astype(str).str.strip()

    # Convert dates
    data["order_date"] = pd.to_datetime(data["order_date"], errors="coerce")
    data["ship_date"] = pd.to_datetime(data["ship_date"], errors="coerce")

    # Create variables if missing
    if "order_year" not in data.columns:
        data["order_year"] = data["order_date"].dt.year

    if "order_month" not in data.columns:
        data["order_month"] = data["order_date"].dt.month

    if "order_month_name" not in data.columns:
        data["order_month_name"] = data["order_date"].dt.month_name()

    if "order_quarter" not in data.columns:
        data["order_quarter"] = "Q" + data["order_date"].dt.quarter.astype(str)

    if "order_year_month" not in data.columns:
        data["order_year_month"] = data["order_date"].dt.to_period("M").astype(str)

    if "shipping_delay_days" not in data.columns:
        data["shipping_delay_days"] = (data["ship_date"] - data["order_date"]).dt.days

    if "profit_margin" not in data.columns:
        data["profit_margin"] = np.where(
            data["sales"] != 0,
            data["profit"] / data["sales"],
            np.nan
        )

    if "profit_status" not in data.columns:
        data["profit_status"] = np.select(
            [data["profit"] > 0, data["profit"] < 0, data["profit"] == 0],
            ["Profitable", "Loss-making", "Break-even"],
            default="Unknown"
        )

    if "discount_band" not in data.columns:
        conditions = [
            data["discount"] == 0,
            (data["discount"] > 0) & (data["discount"] <= 0.10),
            (data["discount"] > 0.10) & (data["discount"] <= 0.20),
            (data["discount"] > 0.20) & (data["discount"] <= 0.30),
            (data["discount"] > 0.30) & (data["discount"] <= 0.50),
            data["discount"] > 0.50
        ]
        choices = [
            "No discount",
            "Low discount",
            "Moderate discount",
            "High discount",
            "Very high discount",
            "Extreme discount"
        ]
        data["discount_band"] = np.select(conditions, choices, default="Unknown")

    if "shipping_cost_ratio" not in data.columns:
        data["shipping_cost_ratio"] = np.where(
            data["sales"] != 0,
            data["shipping_cost"] / data["sales"],
            np.nan
        )

    if "order_size" not in data.columns:
        data["order_size"] = pd.cut(
            data["quantity"],
            bins=[0, 2, 5, 10, np.inf],
            labels=["Small", "Medium", "Large", "Very Large"],
            include_lowest=True
        ).astype(str)

    if "market_group" not in data.columns:
        data["market_group"] = data.get("market", "Unknown")

    # Convert date columns to strings for SQLite compatibility
    data["order_date"] = data["order_date"].dt.strftime("%Y-%m-%d")
    data["ship_date"] = data["ship_date"].dt.strftime("%Y-%m-%d")

    return data


df = prepare_superstore_data(df, DATA_SOURCE)

required_columns = [
    "row_id", "order_id", "order_date", "ship_date", "customer_id", "customer_name",
    "segment", "product_id", "product_name", "category", "sub_category",
    "city", "state", "country", "region", "market", "sales", "profit",
    "quantity", "discount", "shipping_cost", "ship_mode", "order_priority",
    "order_year", "order_month", "order_quarter", "order_year_month",
    "shipping_delay_days", "profit_margin", "profit_status", "discount_band",
    "shipping_cost_ratio", "order_size"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print("Dataset is ready for SQL modeling.")
display(df.head())

Rows: 51,290
Columns: 36
Dataset is ready for SQL modeling.


,row_id,order_id,order_date,ship_date,order_year,order_month,order_month_name,order_quarter,order_year_month,order_week,order_priority,customer_id,customer_name,segment,product_id,product_name,category,sub_category,country,state,city,region,market,market_group,sales,quantity,discount,profit,shipping_cost,ship_mode,shipping_delay_days,profit_margin,profit_status,discount_band,shipping_cost_ratio,order_size
0,36624,CA-2011-130813,2011-01-07,2011-01-09,2011,1,January,Q1,2011-01,2,High,LS-172304,Lycoris Saunders,Consumer,OFF-PA-10002005,Xerox 225,Office Supplies,Paper,United States,California,Los Angeles,West,US,North America,19,3,0.00,9.33,4.37,Second Class,2,0.49,Profitable,No discount,0.23,Medium
1,37033,CA-2011-148614,2011-01-21,2011-01-26,2011,1,January,Q1,2011-01,4,Medium,MV-174854,Mark Van Huff,Consumer,OFF-PA-10002893,"Wirebound Service Call Books, 5 1/2"" x 4""",Office Supplies,Paper,United States,California,Los Angeles,West,US,North America,19,2,0.00,9.29,0.94,Standard Class,5,0.49,Profitable,No discount,0.05,Small
2,31468,CA-2011-118962,2011-08-05,2011-08-09,2011,8,August,Q3,2011-08,32,Medium,CS-121304,Chad Sievert,Consumer,OFF-PA-10000659,"Adams Phone Message Book, Professional, 400 Me...",Office Supplies,Paper,United States,California,Los Angeles,West,US,North America,21,3,0.00,9.84,1.81,Standard Class,4,0.47,Profitable,No discount,0.09,Medium
3,31469,CA-2011-118962,2011-08-05,2011-08-09,2011,8,August,Q3,2011-08,32,Medium,CS-121304,Chad Sievert,Consumer,OFF-PA-10001144,Xerox 1913,Office Supplies,Paper,United States,California,Los Angeles,West,US,North America,111,2,0.00,53.26,4.59,Standard Class,4,0.48,Profitable,No discount,0.04,Small
4,32440,CA-2011-146969,2011-09-29,2011-10-03,2011,9,September,Q3,2011-09,40,High,AP-109154,Arthur Prichep,Consumer,OFF-PA-10002105,Xerox 223,Office Supplies,Paper,United States,California,Los Angeles,West,US,North America,6,1,0.00,3.11,1.32,Standard Class,4,0.52,Profitable,No discount,0.22,Small


## 4. Build a Simple Relational Model

The cleaned dataset is originally a single table.

To practice SQL in a more realistic way, it will be transformed into a simple star-schema style model:

- `fact_sales`
- `dim_customers`
- `dim_products`
- `dim_geography`

This allows the analysis to use `JOIN` operations instead of querying only one flat table.

### Product Key Note

`product_id` is not treated as the only product identifier.

Some product IDs may be associated with multiple product names or descriptions, so a new `product_key` is created using the combination of:

```text
product_id + product_name + category + sub_category
```

In [9]:
df_model = df.copy()

# Create robust keys for dimensions
df_model["product_key"] = (
    df_model["product_id"].astype(str) + " | " +
    df_model["product_name"].astype(str) + " | " +
    df_model["category"].astype(str) + " | " +
    df_model["sub_category"].astype(str)
)

df_model["geography_key"] = (
    df_model["market"].astype(str) + " | " +
    df_model["region"].astype(str) + " | " +
    df_model["country"].astype(str) + " | " +
    df_model["state"].astype(str) + " | " +
    df_model["city"].astype(str)
)

dim_customers = (
    df_model[["customer_id", "customer_name", "segment"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_products = (
    df_model[["product_key", "product_id", "product_name", "category", "sub_category"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_geography = (
    df_model[["geography_key", "market", "market_group", "region", "country", "state", "city"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

fact_sales_columns = [
    "row_id", "order_id", "order_date", "ship_date", "customer_id", "product_key",
    "geography_key", "sales", "profit", "quantity", "discount", "shipping_cost",
    "ship_mode", "order_priority", "order_year", "order_month", "order_month_name",
    "order_quarter", "order_year_month", "order_week", "shipping_delay_days",
    "profit_margin", "profit_status", "discount_band", "shipping_cost_ratio",
    "order_size"
]

# Keep only fact columns that exist in the dataset
fact_sales_columns = [col for col in fact_sales_columns if col in df_model.columns]

fact_sales = df_model[fact_sales_columns].copy()

model_summary = pd.DataFrame({
    "Table": ["fact_sales", "dim_customers", "dim_products", "dim_geography"],
    "Rows": [
        fact_sales.shape[0],
        dim_customers.shape[0],
        dim_products.shape[0],
        dim_geography.shape[0]
    ],
    "Columns": [
        fact_sales.shape[1],
        dim_customers.shape[1],
        dim_products.shape[1],
        dim_geography.shape[1]
    ]
})

model_summary

,Table,Rows,Columns
0,fact_sales,51290,26
1,dim_customers,4873,3
2,dim_products,10768,5
3,dim_geography,3819,7


## 5. Load Tables into SQLite

SQLite is used here because it runs easily in a notebook without external setup.

The same SQL logic can later be adapted to PostgreSQL, MySQL, or SQL Server.

In [10]:
possible_db_dirs = [
    Path("../sql"),
    Path("sql"),
    Path(".")
]

DB_DIR = possible_db_dirs[0]

if not DB_DIR.exists():
    DB_DIR = possible_db_dirs[1]

DB_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "nova_retail.db"

connection = sqlite3.connect(DB_PATH)

fact_sales.to_sql("fact_sales", connection, if_exists="replace", index=False)
dim_customers.to_sql("dim_customers", connection, if_exists="replace", index=False)
dim_products.to_sql("dim_products", connection, if_exists="replace", index=False)
dim_geography.to_sql("dim_geography", connection, if_exists="replace", index=False)

print(f"SQLite database created at: {DB_PATH}")

SQLite database created at: sql/nova_retail.db


In [11]:
tables_check = run_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection,
    title="Tables available in the SQLite database:"
)

Tables available in the SQLite database:


,table_name
0,dim_customers
1,dim_geography
2,dim_products
3,fact_sales


## 6. SQL Analysis Questions

The following SQL queries validate and deepen the EDA findings.

The queries are grouped by business area.

## 6.1 Overall KPI Summary

### Business Question

**Is Nova Retail profitable overall?**

In [12]:
query_kpi_summary = """
SELECT
    COUNT(*) AS order_lines,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS total_customers,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / NULLIF(SUM(sales), 0), 4) AS profit_margin,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_rows,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_making_row_share
FROM fact_sales;
"""

kpi_sql = run_query(query_kpi_summary, connection, "KPI Summary")

KPI Summary


,order_lines,total_orders,total_customers,total_sales,total_profit,profit_margin,loss_making_rows,loss_making_row_share
0,51290,25035,4873,"12,642,905.00","1,467,457.29",0.12,12544,0.24


### Interpretation

This query checks whether the company is profitable overall and how many transactions are loss-making.

A high share of loss-making rows means that overall profit may be hiding important business issues.

## 6.2 Yearly Sales and Profit Growth

### Business Question

**Are sales and profit growing at the same pace?**

In [13]:
query_yearly_growth = """
WITH yearly_performance AS (
    SELECT
        order_year,
        ROUND(SUM(sales), 2) AS sales,
        ROUND(SUM(profit), 2) AS profit,
        ROUND(SUM(profit) / NULLIF(SUM(sales), 0), 4) AS profit_margin
    FROM fact_sales
    GROUP BY order_year
),
yearly_with_lag AS (
    SELECT
        order_year,
        sales,
        profit,
        profit_margin,
        LAG(sales) OVER (ORDER BY order_year) AS previous_year_sales,
        LAG(profit) OVER (ORDER BY order_year) AS previous_year_profit
    FROM yearly_performance
)
SELECT
    order_year,
    sales,
    profit,
    profit_margin,
    ROUND((sales - previous_year_sales) / NULLIF(previous_year_sales, 0), 4) AS sales_growth,
    ROUND((profit - previous_year_profit) / NULLIF(previous_year_profit, 0), 4) AS profit_growth
FROM yearly_with_lag
ORDER BY order_year;
"""

yearly_growth_sql = run_query(query_yearly_growth, connection, "Yearly Sales and Profit Growth")

Yearly Sales and Profit Growth


,order_year,sales,profit,profit_margin,sales_growth,profit_growth
0,2011,"2,259,511.00","248,940.81",0.11,NaN,NaN
1,2012,"2,677,493.00","307,415.28",0.11,0.18,0.23
2,2013,"3,405,860.00","406,935.23",0.12,0.27,0.32
3,2014,"4,300,041.00","504,165.97",0.12,0.26,0.24


### Interpretation

This query uses a CTE and `LAG()` window function to compare each year with the previous year.

If sales growth is higher than profit growth, the business may be growing in volume but not in efficiency.

## 6.3 Monthly Running Totals

### Business Question

**How do cumulative sales and cumulative profit evolve over time?**

In [14]:
query_monthly_running_total = """
WITH monthly_performance AS (
    SELECT
        order_year_month,
        ROUND(SUM(sales), 2) AS sales,
        ROUND(SUM(profit), 2) AS profit
    FROM fact_sales
    GROUP BY order_year_month
)
SELECT
    order_year_month,
    sales,
    profit,
    ROUND(SUM(sales) OVER (ORDER BY order_year_month), 2) AS running_sales,
    ROUND(SUM(profit) OVER (ORDER BY order_year_month), 2) AS running_profit
FROM monthly_performance
ORDER BY order_year_month;
"""

monthly_running_total_sql = run_query(query_monthly_running_total, connection, "Monthly Running Totals")

Monthly Running Totals


,order_year_month,sales,profit,running_sales,running_profit
0,2011-01,"98,902.00","8,321.80","98,902.00","8,321.80"
1,2011-02,"91,152.00","12,417.91","190,054.00","20,739.71"
2,2011-03,"145,726.00","15,303.57","335,780.00","36,043.28"
3,2011-04,"116,927.00","12,902.32","452,707.00","48,945.60"
4,2011-05,"146,762.00","12,183.83","599,469.00","61,129.43"
5,2011-06,"215,214.00","23,415.25","814,683.00","84,544.68"
6,2011-07,"115,518.00","5,585.00","930,201.00","90,129.68"
7,2011-08,"207,570.00","23,713.67","1,137,771.00","113,843.35"
8,2011-09,"290,230.00","35,776.88","1,428,001.00","149,620.23"
9,2011-10,"199,070.00","25,963.42","1,627,071.00","175,583.65"


### Interpretation

Running totals help show whether profitability accumulates consistently over time or whether certain periods reduce the overall trajectory.

## 6.4 Category and Sub-Category Profitability

### Business Question

**Which categories and sub-categories create or destroy business value?**

In [15]:
query_category_performance = """
SELECT
    p.category,
    ROUND(SUM(f.sales), 2) AS sales,
    ROUND(SUM(f.profit), 2) AS profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
    SUM(f.quantity) AS quantity,
    COUNT(DISTINCT f.order_id) AS orders,
    SUM(CASE WHEN f.profit < 0 THEN 1 ELSE 0 END) AS loss_making_rows
FROM fact_sales f
JOIN dim_products p
    ON f.product_key = p.product_key
GROUP BY p.category
ORDER BY profit DESC;
"""

category_performance_sql = run_query(query_category_performance, connection, "Category Performance")

Category Performance


,category,sales,profit,profit_margin,quantity,orders,loss_making_rows
0,Technology,"4,744,691.00","663,778.73",0.14,35176,8354,2424
1,Office Supplies,"3,787,330.00","518,473.83",0.14,108182,19003,7003
2,Furniture,"4,110,884.00","285,204.72",0.07,34954,8195,3117


In [16]:
query_sub_category_performance = """
SELECT
    p.category,
    p.sub_category,
    ROUND(SUM(f.sales), 2) AS sales,
    ROUND(SUM(f.profit), 2) AS profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
    SUM(f.quantity) AS quantity,
    COUNT(DISTINCT f.order_id) AS orders,
    SUM(CASE WHEN f.profit < 0 THEN 1 ELSE 0 END) AS loss_making_rows,
    RANK() OVER (ORDER BY SUM(f.profit) DESC) AS profit_rank
FROM fact_sales f
JOIN dim_products p
    ON f.product_key = p.product_key
GROUP BY p.category, p.sub_category
ORDER BY profit ASC;
"""

sub_category_performance_sql = run_query(query_sub_category_performance, connection, "Sub-Category Performance")

Sub-Category Performance


,category,sub_category,sales,profit,profit_margin,quantity,orders,loss_making_rows,profit_rank
0,Furniture,Tables,"757,034.00","-64,083.39",-0.08,3083,836,496,17
1,Office Supplies,Fasteners,"83,254.00","11,525.42",0.14,8390,2304,612,16
2,Office Supplies,Labels,"73,433.00","15,010.51",0.20,9322,2460,520,15
3,Office Supplies,Supplies,"243,090.00","22,583.26",0.09,8543,2281,589,14
4,Office Supplies,Envelopes,"170,926.00","29,601.12",0.17,8380,2310,555,13
5,Furniture,Furnishings,"385,609.00","46,967.43",0.12,11225,2965,804,12
6,Office Supplies,Art,"372,163.00","57,953.91",0.16,16301,4366,910,11
7,Technology,Machines,"779,071.00","58,867.87",0.08,4906,1422,454,10
8,Office Supplies,Paper,"244,307.00","59,207.68",0.24,12822,3234,527,9
9,Office Supplies,Binders,"461,952.00","72,449.85",0.16,21429,5392,1528,8


### Interpretation

This analysis helps identify whether revenue and profit are aligned.

A sub-category with high sales but negative profit should be investigated further.

## 6.5 Product Ranking

### Business Question

**Which products are the strongest and weakest contributors to profit?**

In [17]:
query_top_products_by_profit = """
WITH product_performance AS (
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        p.sub_category,
        ROUND(SUM(f.sales), 2) AS sales,
        ROUND(SUM(f.profit), 2) AS profit,
        ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
        COUNT(DISTINCT f.order_id) AS orders
    FROM fact_sales f
    JOIN dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.product_id, p.product_name, p.category, p.sub_category
),
ranked_products AS (
    SELECT
        *,
        RANK() OVER (ORDER BY profit DESC) AS profit_rank,
        RANK() OVER (ORDER BY sales DESC) AS sales_rank
    FROM product_performance
)
SELECT
    product_id,
    product_name,
    category,
    sub_category,
    sales,
    profit,
    profit_margin,
    orders,
    profit_rank,
    sales_rank
FROM ranked_products
WHERE profit_rank <= 10
ORDER BY profit_rank;
"""

top_products_by_profit_sql = run_query(query_top_products_by_profit, connection, "Top 10 Products by Profit")

Top 10 Products by Profit


,product_id,product_name,category,sub_category,sales,profit,profit_margin,orders,profit_rank,sales_rank
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,"61,600.00","25,199.93",0.41,5,1,1
1,OFF-AP-10004512,"Hoover Stove, Red",Office Supplies,Appliances,"21,148.00","10,345.58",0.49,7,2,8
2,TEC-PH-10004823,"Nokia Smart Phone, Full Size",Technology,Phones,"22,261.00","8,121.48",0.36,11,3,5
3,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,"27,454.00","7,753.04",0.28,10,4,3
4,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,"18,840.00","6,983.88",0.37,8,5,11
5,FUR-CH-10002250,"Office Star Executive Leather Armchair, Black",Furniture,Chairs,"15,289.00","6,123.26",0.40,10,6,29
6,TEC-PH-10004664,"Nokia Smart Phone, with Caller ID",Technology,Phones,"30,042.00","5,455.95",0.18,10,7,2
7,OFF-AP-10002330,"Hamilton Beach Stove, Silver",Office Supplies,Appliances,"18,248.00","5,452.46",0.30,7,8,14
8,TEC-PH-10000303,"Samsung Smart Phone, VoIP",Technology,Phones,"16,797.00","5,356.81",0.32,6,9,21
9,FUR-CH-10002203,"SAFCO Executive Leather Armchair, Black",Furniture,Chairs,"12,301.00","5,003.10",0.41,5,10,53


In [18]:
query_loss_making_products = """
WITH product_performance AS (
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        p.sub_category,
        ROUND(SUM(f.sales), 2) AS sales,
        ROUND(SUM(f.profit), 2) AS profit,
        ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
        COUNT(DISTINCT f.order_id) AS orders
    FROM fact_sales f
    JOIN dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.product_id, p.product_name, p.category, p.sub_category
)
SELECT
    product_id,
    product_name,
    category,
    sub_category,
    sales,
    profit,
    profit_margin,
    orders
FROM product_performance
ORDER BY profit ASC
LIMIT 20;
"""

loss_making_products_sql = run_query(query_loss_making_products, connection, "Top 20 Loss-Making Products")

Top 20 Loss-Making Products


,product_id,product_name,category,sub_category,sales,profit,profit_margin,orders
0,TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,Technology,Machines,"11,100.00","-8,879.97",-0.80,3
1,OFF-AP-10001623,"Hoover Stove, White",Office Supplies,Appliances,"11,730.00","-4,958.16",-0.42,6
2,TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,Technology,Machines,"16,830.00","-4,589.97",-0.27,4
3,TEC-PH-10002991,"Apple Smart Phone, Full Size",Technology,Phones,"7,258.00","-4,574.64",-0.63,4
4,TEC-MOT-10003050,"Motorola Smart Phone, Cordless",Technology,Phones,"10,350.00","-3,998.68",-0.39,7
5,TEC-MA-10004125,Cubify CubeX 3D Printer Triple Head Print,Technology,Machines,"8,000.00","-3,839.99",-0.48,1
6,FUR-CH-10001582,"Office Star Executive Leather Armchair, Black",Furniture,Chairs,"6,498.00","-3,066.78",-0.47,5
7,FUR-TA-10000198,Chromcraft Bull-Nose Wood Oval Conference Tabl...,Furniture,Tables,"9,918.00","-2,876.12",-0.29,5
8,FUR-TA-10002885,"Lesro Computer Table, Fully Assembled",Furniture,Tables,"1,200.00","-2,798.49",-2.33,3
9,FUR-TA-10002172,"Hon Conference Table, Rectangular",Furniture,Tables,"3,667.00","-2,619.31",-0.71,3


### Interpretation

The most profitable products and the best-selling products are not always the same.

This is why profitability should be evaluated with both sales and margin.

## 6.6 High Sales but Low Margin Products

### Business Question

**Which products sell well but do not generate enough profit?**

In [19]:
query_high_sales_low_margin_products = """
WITH product_performance AS (
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        p.sub_category,
        SUM(f.sales) AS sales,
        SUM(f.profit) AS profit,
        SUM(f.profit) / NULLIF(SUM(f.sales), 0) AS profit_margin,
        COUNT(DISTINCT f.order_id) AS orders
    FROM fact_sales f
    JOIN dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.product_id, p.product_name, p.category, p.sub_category
),
overall_margin AS (
    SELECT
        SUM(profit) / NULLIF(SUM(sales), 0) AS company_profit_margin
    FROM fact_sales
),
average_product_sales AS (
    SELECT
        AVG(sales) AS avg_product_sales
    FROM product_performance
)
SELECT
    pp.product_id,
    pp.product_name,
    pp.category,
    pp.sub_category,
    ROUND(pp.sales, 2) AS sales,
    ROUND(pp.profit, 2) AS profit,
    ROUND(pp.profit_margin, 4) AS profit_margin,
    pp.orders
FROM product_performance pp
CROSS JOIN overall_margin om
CROSS JOIN average_product_sales aps
WHERE pp.sales > aps.avg_product_sales
  AND pp.profit_margin < om.company_profit_margin
ORDER BY pp.sales DESC
LIMIT 20;
"""

high_sales_low_margin_products_sql = run_query(
    query_high_sales_low_margin_products,
    connection,
    "High Sales but Low Margin Products"
)

High Sales but Low Margin Products


,product_id,product_name,category,sub_category,sales,profit,profit_margin,orders
0,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,Technology,Machines,"22,638.00","-1,811.08",-0.08,1
1,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,Chairs,"21,870.00",0.00,0.00,8
2,FUR-CH-10000027,"SAFCO Executive Leather Armchair, Black",Furniture,Chairs,"21,329.00","1,363.23",0.06,12
3,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,"19,824.00","2,233.51",0.11,11
4,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,Office Supplies,Binders,"19,026.00",760.98,0.04,9
5,TEC-PH-10001457,"Apple Smart Phone, Full Size",Technology,Phones,"18,718.00","1,695.30",0.09,8
6,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,"17,965.00","-1,878.17",-0.10,6
7,FUR-BO-10004679,"Safco Library with Doors, Pine",Furniture,Bookcases,"17,433.00",375.23,0.02,8
8,FUR-CH-10002247,"Hon Executive Leather Armchair, Adjustable",Furniture,Chairs,"17,122.00","1,855.69",0.11,11
9,FUR-BO-10001934,"Bush Library with Doors, Metal",Furniture,Bookcases,"17,093.00",-309.34,-0.02,14


### Interpretation

These products may look successful because they generate high revenue.

However, they may reduce business efficiency if their margins are below the company average.

## 6.7 Customer Segment Performance

### Business Question

**Which customer segments generate the most value?**

In [20]:
query_segment_performance = """
SELECT
    c.segment,
    ROUND(SUM(f.sales), 2) AS sales,
    ROUND(SUM(f.profit), 2) AS profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
    COUNT(DISTINCT f.order_id) AS orders,
    COUNT(DISTINCT f.customer_id) AS customers
FROM fact_sales f
JOIN dim_customers c
    ON f.customer_id = c.customer_id
GROUP BY c.segment
ORDER BY profit DESC;
"""

segment_performance_sql = run_query(query_segment_performance, connection, "Customer Segment Performance")

Customer Segment Performance


,segment,sales,profit,profit_margin,orders,customers
0,Consumer,"6,508,141.00","749,239.78",0.12,13104,2509
1,Corporate,"3,824,808.00","441,208.33",0.12,7673,1457
2,Home Office,"2,309,956.00","277,009.18",0.12,4687,907


In [21]:
query_top_customers = """
WITH customer_performance AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.segment,
        ROUND(SUM(f.sales), 2) AS sales,
        ROUND(SUM(f.profit), 2) AS profit,
        ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
        COUNT(DISTINCT f.order_id) AS orders
    FROM fact_sales f
    JOIN dim_customers c
        ON f.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name, c.segment
)
SELECT
    *,
    RANK() OVER (ORDER BY profit DESC) AS profit_rank
FROM customer_performance
ORDER BY profit DESC
LIMIT 20;
"""

top_customers_sql = run_query(query_top_customers, connection, "Top 20 Customers by Profit")

Top 20 Customers by Profit


,customer_id,customer_name,segment,sales,profit,profit_margin,orders,profit_rank
0,TC-209804,Tamara Chand,Corporate,"19,050.00","8,981.32",0.47,5,1
1,RB-193604,Raymond Buch,Consumer,"15,117.00","6,976.10",0.46,6,2
2,SC-200954,Sanjit Chand,Consumer,"14,145.00","5,757.41",0.41,9,3
3,HL-150404,Hunter Lopez,Consumer,"12,875.00","5,622.43",0.44,6,4
4,AB-101054,Adrian Barton,Consumer,"14,476.00","5,444.81",0.38,10,5
5,SP-209202,Susan Pistek,Consumer,"16,566.00","4,974.51",0.30,12,6
6,TA-213854,Tom Ashbrook,Home Office,"14,596.00","4,703.79",0.32,4,7
7,CA-127751,Cynthia Arntzen,Consumer,"11,493.00","4,045.88",0.35,5,8
8,PJ-188352,Patrick Jones,Corporate,"8,509.00","3,986.00",0.47,3,9
9,CM-123854,Christopher Martinez,Consumer,"8,953.00","3,899.89",0.44,4,10


### Interpretation

A customer should not be evaluated only by revenue.

A customer with high sales but low margin may require a different commercial strategy.

## 6.8 Geographic Performance

### Business Question

**Which markets and countries are the most and least profitable?**

In [22]:
query_market_performance = """
SELECT
    g.market,
    ROUND(SUM(f.sales), 2) AS sales,
    ROUND(SUM(f.profit), 2) AS profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
    COUNT(DISTINCT f.order_id) AS orders,
    COUNT(DISTINCT f.customer_id) AS customers
FROM fact_sales f
JOIN dim_geography g
    ON f.geography_key = g.geography_key
GROUP BY g.market
ORDER BY profit DESC;
"""

market_performance_sql = run_query(query_market_performance, connection, "Market Performance")

Market Performance


,market,sales,profit,profit_margin,orders,customers
0,APAC,"3,585,833.00","436,000.05",0.12,5437,796
1,EU,"2,938,139.00","372,829.74",0.13,4593,795
2,US,"2,297,354.00","286,397.02",0.12,5009,793
3,LATAM,"2,164,687.00","221,643.49",0.10,5138,794
4,Africa,"783,776.00","88,871.63",0.11,2232,754
5,EMEA,"806,184.00","43,897.97",0.05,2462,760
6,Canada,"66,932.00","17,817.39",0.27,201,181


In [23]:
query_country_performance = """
SELECT
    g.market,
    g.region,
    g.country,
    ROUND(SUM(f.sales), 2) AS sales,
    ROUND(SUM(f.profit), 2) AS profit,
    ROUND(SUM(f.profit) / NULLIF(SUM(f.sales), 0), 4) AS profit_margin,
    COUNT(DISTINCT f.order_id) AS orders
FROM fact_sales f
JOIN dim_geography g
    ON f.geography_key = g.geography_key
GROUP BY g.market, g.region, g.country
ORDER BY profit ASC
LIMIT 20;
"""

country_performance_sql = run_query(country_performance_sql if False else query_country_performance, connection, "Bottom 20 Countries by Profit")

Bottom 20 Countries by Profit


,market,region,country,sales,profit,profit_margin,orders
0,EMEA,EMEA,Turkey,"108,524.00","-98,447.23",-0.91,632
1,Africa,Africa,Nigeria,"54,347.00","-80,750.72",-1.49,410
2,EU,Central,Netherlands,"77,520.00","-41,070.07",-0.53,204
3,LATAM,Central,Honduras,"90,137.00","-29,482.37",-0.33,349
4,APAC,Central Asia,Pakistan,"58,873.00","-22,446.65",-0.38,123
5,LATAM,South,Argentina,"57,510.00","-18,693.80",-0.33,191
6,LATAM,Central,Panama,"51,556.00","-17,723.45",-0.34,199
7,EU,North,Sweden,"30,490.00","-17,519.37",-0.57,100
8,APAC,Southeast Asia,Philippines,"183,433.00","-16,128.22",-0.09,326
9,APAC,North Asia,South Korea,"33,133.00","-12,792.83",-0.39,82


### Interpretation

Markets with high revenue but weak margin should be reviewed.

The goal is to identify where growth is profitable and where growth may hide losses.

## 6.9 Discount Impact Analysis

### Business Question

**Do discounts improve or reduce profitability?**

In [24]:
query_discount_impact = """
WITH discount_analysis AS (
    SELECT
        CASE
            WHEN discount = 0 THEN 'No discount'
            WHEN discount > 0 AND discount <= 0.10 THEN 'Low discount'
            WHEN discount > 0.10 AND discount <= 0.20 THEN 'Moderate discount'
            WHEN discount > 0.20 AND discount <= 0.30 THEN 'High discount'
            WHEN discount > 0.30 AND discount <= 0.50 THEN 'Very high discount'
            WHEN discount > 0.50 THEN 'Extreme discount'
            ELSE 'Unknown'
        END AS discount_band,
        CASE
            WHEN discount = 0 THEN 1
            WHEN discount > 0 AND discount <= 0.10 THEN 2
            WHEN discount > 0.10 AND discount <= 0.20 THEN 3
            WHEN discount > 0.20 AND discount <= 0.30 THEN 4
            WHEN discount > 0.30 AND discount <= 0.50 THEN 5
            WHEN discount > 0.50 THEN 6
            ELSE 7
        END AS discount_band_order,
        sales,
        profit,
        order_id
    FROM fact_sales
)
SELECT
    discount_band,
    ROUND(SUM(sales), 2) AS sales,
    ROUND(SUM(profit), 2) AS profit,
    ROUND(SUM(profit) / NULLIF(SUM(sales), 0), 4) AS profit_margin,
    COUNT(*) AS order_lines,
    COUNT(DISTINCT order_id) AS orders,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_rows,
    ROUND(
        1.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*),
        4
    ) AS loss_rate
FROM discount_analysis
GROUP BY discount_band, discount_band_order
ORDER BY discount_band_order;
"""

discount_impact_sql = run_query(query_discount_impact, connection, "Discount Impact Analysis")

Discount Impact Analysis


,discount_band,sales,profit,profit_margin,order_lines,orders,loss_making_rows,loss_rate
0,No discount,"6,992,734.00","1,770,695.27",0.25,29009,15211,0,0.00
1,Low discount,"1,962,633.00","338,189.26",0.17,4679,3028,901,0.19
2,Moderate discount,"1,757,296.00","173,254.84",0.10,6274,4363,1463,0.23
3,High discount,"382,540.00","-21,155.61",-0.06,967,843,601,0.62
4,Very high discount,"1,176,078.00","-380,944.82",-0.32,6189,3674,5407,0.87
5,Extreme discount,"371,624.00","-412,581.66",-1.11,4172,2369,4172,1.00


### Interpretation

This query uses `CASE WHEN` to classify discount levels.

If higher discounts have lower margins or higher loss rates, the discount strategy should be reviewed.

## 6.10 Shipping and Operational Performance

### Business Question

**Do shipping methods and order priorities affect profitability?**

In [25]:
query_shipping_performance = """
SELECT
    ship_mode,
    ROUND(SUM(sales), 2) AS sales,
    ROUND(SUM(profit), 2) AS profit,
    ROUND(SUM(profit) / NULLIF(SUM(sales), 0), 4) AS profit_margin,
    ROUND(SUM(shipping_cost), 2) AS shipping_cost,
    ROUND(SUM(shipping_cost) / NULLIF(SUM(sales), 0), 4) AS shipping_cost_ratio,
    ROUND(AVG(shipping_delay_days), 2) AS avg_shipping_delay_days,
    COUNT(DISTINCT order_id) AS orders
FROM fact_sales
GROUP BY ship_mode
ORDER BY profit DESC;
"""

shipping_performance_sql = run_query(query_shipping_performance, connection, "Shipping Mode Performance")

Shipping Mode Performance


,ship_mode,sales,profit,profit_margin,shipping_cost,shipping_cost_ratio,avg_shipping_delay_days,orders
0,Standard Class,"7,578,889.00","890,596.02",0.12,"614,627.66",0.08,5.00,15154
1,Second Class,"2,565,747.00","292,583.53",0.11,"314,111.79",0.12,3.23,5119
2,First Class,"1,831,067.00","208,104.68",0.11,"308,102.54",0.17,2.18,3821
3,Same Day,"667,202.00","76,173.07",0.11,"115,973.72",0.17,0.04,1347


In [26]:
query_priority_performance = """
SELECT
    order_priority,
    ROUND(SUM(sales), 2) AS sales,
    ROUND(SUM(profit), 2) AS profit,
    ROUND(SUM(profit) / NULLIF(SUM(sales), 0), 4) AS profit_margin,
    ROUND(SUM(shipping_cost), 2) AS shipping_cost,
    COUNT(DISTINCT order_id) AS orders
FROM fact_sales
GROUP BY order_priority
ORDER BY profit DESC;
"""

priority_performance_sql = run_query(query_priority_performance, connection, "Order Priority Performance")

Order Priority Performance


,order_priority,sales,profit,profit_margin,shipping_cost,orders
0,Medium,"7,281,105.00","864,203.76",0.12,"542,812.82",14484
1,High,"3,807,699.00","420,373.51",0.11,"509,545.87",7767
2,Critical,"986,258.00","124,224.16",0.13,"234,823.94",1967
3,Low,"567,843.00","58,655.85",0.10,"65,633.08",1212


### Interpretation

Shipping should be evaluated using both cost and profitability.

A shipping method is not necessarily bad because it costs more. It becomes a problem if the additional cost does not support profitable sales.

## 6.11 Pareto Analysis

### Business Question

**How concentrated is profit among products?**

This analysis checks how many profitable products are needed to generate around 80% of total positive profit.

In [27]:
query_pareto_products = """
WITH product_profit AS (
    SELECT
        p.product_id,
        p.product_name,
        p.category,
        p.sub_category,
        SUM(f.profit) AS profit
    FROM fact_sales f
    JOIN dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.product_id, p.product_name, p.category, p.sub_category
    HAVING SUM(f.profit) > 0
),
ranked_products AS (
    SELECT
        *,
        ROW_NUMBER() OVER (ORDER BY profit DESC) AS product_rank,
        COUNT(*) OVER () AS total_profitable_products,
        SUM(profit) OVER (ORDER BY profit DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_profit,
        SUM(profit) OVER () AS total_positive_profit
    FROM product_profit
),
pareto AS (
    SELECT
        product_id,
        product_name,
        category,
        sub_category,
        ROUND(profit, 2) AS profit,
        product_rank,
        total_profitable_products,
        ROUND(cumulative_profit, 2) AS cumulative_profit,
        ROUND(cumulative_profit / NULLIF(total_positive_profit, 0), 4) AS cumulative_profit_share,
        ROUND(1.0 * product_rank / total_profitable_products, 4) AS product_share
    FROM ranked_products
)
SELECT *
FROM pareto
WHERE cumulative_profit_share <= 0.80
ORDER BY product_rank;
"""

pareto_products_sql = run_query(query_pareto_products, connection, "Products Contributing to Around 80% of Positive Profit")

Products Contributing to Around 80% of Positive Profit


,product_id,product_name,category,sub_category,profit,product_rank,total_profitable_products,cumulative_profit,cumulative_profit_share,product_share
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,"25,199.93",1,7691,"25,199.93",0.01,0.00
1,OFF-AP-10004512,"Hoover Stove, Red",Office Supplies,Appliances,"10,345.58",2,7691,"35,545.51",0.02,0.00
2,TEC-PH-10004823,"Nokia Smart Phone, Full Size",Technology,Phones,"8,121.48",3,7691,"43,666.99",0.02,0.00
3,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,"7,753.04",4,7691,"51,420.03",0.03,0.00
4,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,"6,983.88",5,7691,"58,403.91",0.03,0.00
...,...,...,...,...,...,...,...,...,...,...
1951,TEC-PH-10001805,"Nokia Speaker Phone, Full Size",Technology,Phones,252.57,1952,7691,"1,619,874.15",0.80,0.25
1952,FUR-SAF-10000881,"Safco 3-Shelf Cabinet, Metal",Furniture,Bookcases,252.36,1953,7691,"1,620,126.51",0.80,0.25
1953,OFF-AR-10004109,"Boston Sketch Pad, Fluorescent",Office Supplies,Art,252.32,1954,7691,"1,620,378.83",0.80,0.25
1954,OFF-EAT-10001933,"Eaton Cards & Envelopes, Recycled",Office Supplies,Paper,252.29,1955,7691,"1,620,631.13",0.80,0.25


In [28]:
query_pareto_summary = """
WITH product_profit AS (
    SELECT
        p.product_id,
        p.product_name,
        SUM(f.profit) AS profit
    FROM fact_sales f
    JOIN dim_products p
        ON f.product_key = p.product_key
    GROUP BY p.product_id, p.product_name
    HAVING SUM(f.profit) > 0
),
ranked_products AS (
    SELECT
        *,
        ROW_NUMBER() OVER (ORDER BY profit DESC) AS product_rank,
        COUNT(*) OVER () AS total_profitable_products,
        SUM(profit) OVER (ORDER BY profit DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_profit,
        SUM(profit) OVER () AS total_positive_profit
    FROM product_profit
),
first_rank_reaching_80pct AS (
    SELECT
        MIN(product_rank) AS products_needed_for_80pct_profit
    FROM ranked_products
    WHERE cumulative_profit / NULLIF(total_positive_profit, 0) >= 0.80
)
SELECT
    products_needed_for_80pct_profit,
    total_profitable_products,
    ROUND(1.0 * products_needed_for_80pct_profit / total_profitable_products, 4) AS share_of_profitable_products
FROM first_rank_reaching_80pct
CROSS JOIN (
    SELECT COUNT(*) AS total_profitable_products FROM product_profit
);
"""

pareto_summary_sql = run_query(query_pareto_summary, connection, "Pareto Summary")

Pareto Summary


,products_needed_for_80pct_profit,total_profitable_products,share_of_profitable_products
0,1957,7691,0.25


### Interpretation

If a small share of products generates most of the profit, the company should protect and prioritize those products.

This can guide product strategy, promotion decisions, and inventory focus.

## 7. SQL Key Findings

This section summarizes the main findings from the SQL analysis.

The values below are generated from the SQL query outputs.

In [29]:
# Extract a few values from SQL output tables for a simple findings summary
overall_margin = kpi_sql.loc[0, "profit_margin"]
loss_share = kpi_sql.loc[0, "loss_making_row_share"]

top_category = category_performance_sql.sort_values("profit", ascending=False).iloc[0]
lowest_sub_category = sub_category_performance_sql.sort_values("profit", ascending=True).iloc[0]

top_market = market_performance_sql.sort_values("profit", ascending=False).iloc[0]
lowest_market = market_performance_sql.sort_values("profit", ascending=True).iloc[0]

best_discount_band = discount_impact_sql.sort_values("profit_margin", ascending=False).iloc[0]
worst_discount_band = discount_impact_sql.sort_values("profit_margin", ascending=True).iloc[0]

sql_key_findings = pd.DataFrame({
    "Business Area": [
        "Overall Profitability",
        "Loss-Making Transactions",
        "Product Category",
        "Sub-Category Risk",
        "Market Performance",
        "Market Performance",
        "Discount Strategy",
        "Discount Strategy"
    ],
    "SQL Finding": [
        f"Overall profit margin is {overall_margin:.2%}.",
        f"{loss_share:.2%} of order lines are loss-making.",
        f"The most profitable category is {top_category['category']}.",
        f"The weakest sub-category by profit is {lowest_sub_category['sub_category']}.",
        f"The most profitable market is {top_market['market']}.",
        f"The weakest market by profit is {lowest_market['market']}.",
        f"The best discount band by margin is {best_discount_band['discount_band']}.",
        f"The worst discount band by margin is {worst_discount_band['discount_band']}."
    ]
})

sql_key_findings

,Business Area,SQL Finding
0,Overall Profitability,Overall profit margin is 11.61%.
1,Loss-Making Transactions,24.46% of order lines are loss-making.
2,Product Category,The most profitable category is Technology.
3,Sub-Category Risk,The weakest sub-category by profit is Tables.
4,Market Performance,The most profitable market is APAC.
5,Market Performance,The weakest market by profit is Canada.
6,Discount Strategy,The best discount band by margin is No discount.
7,Discount Strategy,The worst discount band by margin is Extreme d...


## 8. Export SQL Outputs

The main SQL query results are exported for documentation, reporting, and dashboard preparation.

In [30]:
possible_output_dirs = [
    Path("../reports/sql_outputs"),
    Path("reports/sql_outputs"),
    Path("sql_outputs")
]

OUTPUT_DIR = possible_output_dirs[0]

if not OUTPUT_DIR.parent.exists():
    OUTPUT_DIR = possible_output_dirs[1]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sql_exports = {
    "sql_kpi_summary.csv": kpi_sql,
    "sql_yearly_growth.csv": yearly_growth_sql,
    "sql_monthly_running_total.csv": monthly_running_total_sql,
    "sql_category_performance.csv": category_performance_sql,
    "sql_sub_category_performance.csv": sub_category_performance_sql,
    "sql_top_products_by_profit.csv": top_products_by_profit_sql,
    "sql_loss_making_products.csv": loss_making_products_sql,
    "sql_high_sales_low_margin_products.csv": high_sales_low_margin_products_sql,
    "sql_segment_performance.csv": segment_performance_sql,
    "sql_top_customers.csv": top_customers_sql,
    "sql_market_performance.csv": market_performance_sql,
    "sql_country_performance.csv": country_performance_sql,
    "sql_discount_impact.csv": discount_impact_sql,
    "sql_shipping_performance.csv": shipping_performance_sql,
    "sql_priority_performance.csv": priority_performance_sql,
    "sql_pareto_products.csv": pareto_products_sql,
    "sql_pareto_summary.csv": pareto_summary_sql,
    "sql_key_findings.csv": sql_key_findings
}

for filename, table in sql_exports.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

print(f"SQL output tables exported to: {OUTPUT_DIR}")

SQL output tables exported to: reports/sql_outputs


## 9. Conclusion

The SQL analysis validates and deepens the insights found during the EDA phase.

This phase demonstrated how SQL can be used to answer business questions about:

- overall profitability
- sales and profit growth
- product performance
- loss-making products
- customer segments
- geographic performance
- discount impact
- shipping performance
- Pareto concentration

The next step is to use these findings to design the dashboard and final business recommendations.
